# MR-LFADS Configuration Setup

This tutorial describes how to configure the MR-LFADS inference pipeline.

The configuration is organized hierarchically using Hydra:
```text
main.yaml
├── model/model.yaml
├── datamodule/datamodule.yaml
└── callbacks/callbacks.yaml
```

When going through this tutorial, it may be helpful to locate an example in the `examples/` folder for step-by-step comparison.

## 1. Main Configuration File

The entry point for setup is the `main.yaml` file. 

It points to the location of the `model`, `datamodule` and `callback` YAMLs, and includes other settings such as trainer specifications, tensorboard logging, and seed. 

## 2. Model Configuration File

The model configuration consists of three main categories of parameters:

### (1). Data Specification

These parameters define the structure of the input data, for example:

* `num_other_areas`: total number of areas − 1
* `seq_len`: number of time steps per trial
* `ic_enc_seq_len`: number of time steps used to infer the initial condition (IC)

### (2). Model Architecture

These parameters define the structure of the SRLFADS model:

* `areas_params`: specifies architecture details such as 
    - generator dimensionality
    - inter-area communication dimensions
    - output distribution

For the full list of available parameters, refer to `mrlfads.model.SRLFADS`.

### (3). Regularization

These parameters control regularization, which are key hyperparameters for MR-LFADS:

* L2 regularization (`l2_*`): weight decay applied to GRU parameters
* KL regularization (`kl_*`): regularization on inferred inputs and inter-area messages

For the full list of available parameters, refer to `mrlfads.model.MRLFADS`.

## 3. Datamodule Configuration File

The following parameters are required for configuring the datamodule:

* `filename`: path to the `data.h5` file (relative to `config.paths.datapath`)
* `area_names`: list of brain area names corresponding to the datasets (e.g., ["M1", "PMd"])
* `session_idxs`: list of session indices to include (e.g., [0] for the first session)
* `time_dim`: number of time steps per trial

For additional parameters and options, see `mrlfads.datamodules.BasicDataModule`.

## 4. Callbacks Configuration File

Callbacks are managed via the `mrlfads.callbacks.OnEpochEndCalls` class, which executes specified callback functions at the end of each training or validation epoch. This enables logging, visualization, and intermediate analysis during training, and can be easily extended with custom callbacks. The following callbacks are provided in `mrlfads/callbacks.py`:

* `Log`: Logs training statistics such as learning rate
* `InferredRatesPlot`: Plots inferred firing rates alongside ground-truth spike data (optionally smoothed for comparison)
* `InferredPredsPlot`: Similar to `InferredRatesPlot`, but for held-out neurons
* `ProctorSummaryPlot`: Visualizes training and validation metrics (e.g., loss, KL divergence)
* `AnatomyPlot`: Displays the inferred inter-area communication structure

## 5. Test Run

To run a test configuration:

In [ ]:
import os
import config.paths as path

# Path to the folder that contains main.yaml, relative to config.paths.homepath
# For example: os.path.join(path.homepath, "examples", "01_memory_network")
workdir = NotImplemented 

In [ ]:
import subprocess

subprocess.run(["python", "exec.py"], cwd=workdir, check=True)

To run the examples in `examples/`, first download the corresponding datasets from Zenodo:

1. Memory Network: https://zenodo.org/records/19444085

After downloading, place the dataset folder (e.g., `memory_network_icml2025/data.h5`) into the directory specified by `datapath` in `config/paths.py`.

Once the data is in place, you can run the examples as described above.